In [1]:
from datasets import load_dataset,load_from_disk
import ollama
import os
import glob
import re

/Users/kunkerdthaisong/Llamalama_II/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

#ds = load_dataset("openthaigpt/thai-onet-m6-exam", "english",split="train+test")
#ds.save_to_disk("C:\\Users\\User\\code\\Llamalama_II\\datasets\\m6_eng")
ds=load_from_disk("/Users/kunkerdthaisong/Llamalama_II/ielts/datasets/m6_eng")
def get_llm_response(prompt,choice, system_prompt="You are a helpful assistant and You are a knowledgeable and patient English teacher with expertise in grammar, vocabulary, pronunciation, and conversational skills."):
    
    response = ollama.chat(model='phi3', messages=[
    {
        'role': 'system',
        'content': """You are a knowledgeable and patient English teacher with expertise in grammar, vocabulary, pronunciation, and conversational skills you must answer only in <answer><answer>\
        here are the example:<question>"Dialog1 : At Rose's House\n              Rose :  Are you superstitious?\n              Suda : _1_\n              Rose : Er..it's a belief based on fear or false ideas. For example,\n                          many people believe that Friday the 13th is an unlucky\n                          day. _2_\n             Suda : Well, I _3_ I'm a bit superstitious then. I had a car\n                         accident last Friday and it was Friday the13th.\n             Rose : Did you get hurt?\n             Suda : _4_ How about some coffee?\n             Rose : _5_"<question>\n\n
        <choices>1. I don't agree.\n2. I know exactly.\n3. What's that?\n4. How do you spell it?\n5. What's new?<choices>\n\n
        <answer>3<answer>""",
    },
    {
        'role': 'user',
        'content':f"""<question>{prompt}<question>\n\n
        <choices>{choice}<choices>\n\n
        <answer><answer>""",
    }
])  
    return response['message']['content']


In [3]:
def get_llm_response(prompt, choices, system_prompt="You are a smart at English language.think carefuly before you answer number.no exlanations and no repeat the question just answer only one number remember answer only one number from choices",no_quesiton=None):
    
    response = ollama.chat(model='llama3.1', messages=[
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': f"""
            question no :{no_quesiton}\n{prompt}
            here the choices:
            {choices}
            """
        }
    ])  
    return response['message']['content']


In [4]:
def catch_case(response_text):
    
    pattern = r"(\d+)"

    match = re.search(pattern, response_text, re.DOTALL)
    
    if match:
        number = match.group(1)
        return int(number)
    else:
        return None

In [18]:
coutn_correct=0
for num in range(53,len(ds)):#len(ds)
    ans=get_llm_response(ds[num]["instruction"],choices=ds[num]["input"],no_quesiton=ds[num]['"no"'])
    no_ans=catch_case(ans)
    no_res=catch_case(ds[num]['result'])
    if no_ans is not None and no_res is not None:
        if int(no_ans) == int(no_ans):
            coutn_correct+=1
            del no_res
            del no_ans

In [20]:
coutn_correct

95

In [17]:
num

53

In [7]:
import json
import os

def parse_text_to_jsonl(input_folder, output_file):
    with open(output_file, 'w') as jsonl_file:
        # Iterate over each .txt file in the input folder
        for filename in sorted(os.listdir(input_folder)):
            if filename.endswith(".txt"):
                file_path = os.path.join(input_folder, filename)
                
                with open(file_path, 'r') as file:
                    lines = file.readlines()
                
                # Initialize data structure for this file
                data = {
                    "text": "",
                    "questions": [],
                    "choices": [],
                    "answers": []
                }
                
                section = "text"
                for line in lines:
                    line = line.strip()
                    
                    if line == "#questions":
                        section = "questions"
                    elif line == "#choices":
                        section = "choices"
                    elif line == "#answers":
                        section = "answers"
                    else:
                        if section == "text":
                            data["text"] += line + " "
                        elif section == "questions":
                            data["questions"].append(line)
                        elif section == "choices":
                            data["choices"].append(line)
                        elif section == "answers":
                            question_id, answer = line.split()
                            data["answers"].append({"question_id": question_id, "answer": answer})

                # Write parsed data as a JSON object in JSONL format
                jsonl_file.write(json.dumps(data) + "\n")

# Usage:
# Specify the folder containing 1.txt, 2.txt, etc. and the output .jsonl file
parse_text_to_jsonl("/Users/kunkerdthaisong/Llamalama_II/ielts/datasets/IELST_example/IELST_reading/", "output_1.jsonl")


ValueError: too many values to unpack (expected 2)

In [6]:
coutn_correct #58+39/140

39

In [113]:
ds[num]

{'year': 2020,
 '"no"': '"7"',
 'instruction': "Dialog 2 : On the coach\n                Tourassistant : Ladies and gentlemen. _6_ before we board for\n                                         the cruise.\n                Tourist :            Sorry to interrupt, but _7_ I can't hear you.\n                Tourassistant : Certainly. _8_ to always have your passport with\n                                         you or else you won't be allowed to get on the boat.\n                 Tourist :           _9_ I don't have my passport. What should I do?\n                 Tourassistant : (Sighing) _10_ ..",
 'input': '1. do you mind making a loud noise?\n2. would you like to speak out?\n3. could you please speak a little louder?\n4. can you lower your volume?\n5. do you want to break in?',
 'result': '3. could you please speak a little louder?',
 'isAnswerable': True,
 'isMultipleChoice': True,
 'isSingleChoiceSolution': True}

In [119]:
len(ds)

220